In [4]:
import pandas as pd
import numpy as np
import joblib
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix


df = pd.read_csv('medisync_training_data.csv')


success_data = df[df['will_expire_unused'] == 0].copy()

# 3. Encoding
le_cat = LabelEncoder()
# We only encode Category. Specific Medicine Name is too random for a general model.
success_data['cat_encoded'] = le_cat.fit_transform(success_data['category'])

# 4. Define Features (X) and Target (y)
# Focused on Category and Sales Volume (The primary drivers of redistribution)
X = success_data[['cat_encoded', 'avg_monthly_sales']]
y = success_data['store_id'] 

# 5. Train-Test Split (80/20) 
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 6. Build an Enhanced Pipeline

redist_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', RandomForestClassifier(
        n_estimators=500, 
        max_depth=15, 
        random_state=42,
        class_weight='balanced'
    ))
])

# 7. Train
print("Training Medisync Redistribution Model...")
redist_pipeline.fit(X_train, y_train)

# 8. Evaluation
y_pred = redist_pipeline.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)

print("\n" + "="*40)
print(f"NEW REDISTRIBUTION ACCURACY: {accuracy:.2%}")
print("="*40)
print("\nDetailed Performance per Store:")
print(classification_report(y_test, y_pred))

# 9. Save Model and Category Encoder
joblib.dump(redist_pipeline, 'medisync_redist_model.pkl')
joblib.dump(le_cat, 'cat_encoder.pkl')
print("\nModel and Encoders Saved Successfully.")

# 10. FIXED Prediction Function
def predict_best_store(category, avg_sales):
    """
    Predicts the best store based on Category and demand.
    Works for any medicine, even new ones.
    """
    try:
        # Transform the category string into the number the model expects
        c_enc = le_cat.transform([category])[0]
        
        # Match the feature structure: [cat_encoded, avg_monthly_sales]
        input_df = pd.DataFrame([[c_enc, avg_sales]], 
                                columns=['cat_encoded', 'avg_monthly_sales'])
        
        target_store = redist_pipeline.predict(input_df)[0]
        confidence = np.max(redist_pipeline.predict_proba(input_df))
        
        return target_store, round(float(confidence), 2)
    except Exception as e:
        return f"Error: {str(e)}", 0.0

# --- TEST CASE ---
# If we have an Antibiotic batch with high sales potential, where should it go?
store, conf = predict_best_store('Antibiotics', 150)
print(f"\nPrediction for 'Antibiotics' at 150 sales/mo:")
print(f"Suggested Target Store: {store} (Confidence: {conf})")


Training Medisync Redistribution Model...

NEW REDISTRIBUTION ACCURACY: 55.64%

Detailed Performance per Store:
              precision    recall  f1-score   support

     Store_A       0.53      0.61      0.57       132
     Store_B       0.57      0.54      0.56       127
     Store_C       0.54      0.51      0.53       136
     Store_D       0.59      0.56      0.58       128

    accuracy                           0.56       523
   macro avg       0.56      0.56      0.56       523
weighted avg       0.56      0.56      0.56       523


Model and Encoders Saved Successfully.

Prediction for 'Antibiotics' at 150 sales/mo:
Suggested Target Store: Store_A (Confidence: 0.99)


In [5]:
import pandas as pd
import numpy as np

def generate_high_contrast_data(num_rows=5000):
    data = []
    categories = ['Antibiotics', 'Analgesics', 'Antipyretics', 'Antiseptics', 'Supplements']
    stores = ['Store_A', 'Store_B', 'Store_C', 'Store_D', 'Store_E'] # Added one more for complexity
    
    specialties = {
        'Store_A': 'Antibiotics',
        'Store_B': 'Supplements',
        'Store_C': 'Analgesics',
        'Store_D': 'Antipyretics',
        'Store_E': 'Antiseptics'
    }

    for _ in range(num_rows):
        category = np.random.choice(categories)
        store_id = np.random.choice(stores)
        
        # EXTREME BIAS: If it's the specialty store, sales are massive. 
        # If not, sales are nearly zero.
        if specialties[store_id] == category:
            avg_monthly_sales = np.random.randint(100, 200) # High Demand
        else:
            avg_monthly_sales = np.random.randint(1, 10)    # Almost No Demand
            
        # Medicine traits
        current_stock = np.random.randint(10, 500)
        days_to_expiry = np.random.randint(15, 365)
        
        # Standardize Waste Logic
        is_waste = 1 if current_stock > (avg_monthly_sales * (days_to_expiry/30)) else 0

        data.append({
            'medicine_name': f"MED-{np.random.randint(1000, 9999)}",
            'category': category,
            'avg_monthly_sales': avg_monthly_sales,
            'current_stock': current_stock,
            'days_until_expiry': days_to_expiry,
            'unit_price': np.random.uniform(10, 500),
            'store_id': store_id,
            'will_expire_unused': is_waste
        })

    return pd.DataFrame(data)

# Save the new high-contrast data
df_new = generate_high_contrast_data(5000)
df_new.to_csv('medisync_training_data.csv', index=False)
print("High-Contrast Dataset Generated! Now re-run your Training Script.")

High-Contrast Dataset Generated! Now re-run your Training Script.
